# Spark

**Seong Cho** · ACME Data Science Essentials

**Lab objective:** use Apache Spark, through PySpark, to process data in parallel with DataFrames and RDDs: filtering and grouping a table, counting words, and a Monte Carlo estimate of $\pi$.

<sub>Lab from the *Data Science Essentials* course of [Foundations of Applied Mathematics](https://foundations-of-applied-mathematics.github.io/) (BYU ACME), © Dr. J. Humpherys, used under [CC BY 3.0 US](http://creativecommons.org/licenses/by/3.0/us/). This lab is not in the published *Data Science Essentials* manual: problem statements are quoted from the course's lab template, and the notes are condensed from it and from my code. Solutions are my own.</sub>

> **Note.** This notebook was not re-run for this archive: it needs PySpark and Java.

## Spark and PySpark

- Spark distributes data and computation across the nodes of a cluster (or the cores of one machine). PySpark is its Python interface.
- A `SparkSession` (`SparkSession.builder.getOrCreate()`) is the entry point for DataFrames; a `SparkContext` is the entry point for RDDs. Either should be stopped with `.stop()` when finished.
- **DataFrames** are distributed tables. `spark.read.option("header", True).csv(file)` reads a CSV; `filter`, `select`, `groupBy(...).count()` and `orderBy` work much like their SQL and pandas counterparts.
- **RDDs** (resilient distributed datasets) are distributed collections of Python objects. Transformations like `map` and `flatMap` are lazy; actions like `collect` and `reduce` trigger the computation. `sc.parallelize(data, partitions)` splits local data into partitions.
- Data for this lab: `mathematicians.csv` (name, country, gender, birth year) and `yoda.txt` (lines of text).

In [ ]:
# solutions.py
# Volume 3: Spark.
# <Name>
# <Class>
# <Date>


import pyspark
from pyspark.sql import SparkSession
import numpy as np

## Problem 1: Female mathematicians of the 19th century

> **Problem 1.** Reads in a csv file of mathematicians and reduces it to the names
> of the female mathemticians born in the 19th century. Returns a list
> of the first 5 of these names.

In [ ]:
# Problem 1
def prob1(filename='mathematicians.csv'):
    """Reads in a csv file of mathematicians and reduces it to the names
    of the female mathemticians born in the 19th century. Returns a list
    of the first 5 of these names.
    """
    spark = SparkSession.builder.getOrCreate()
    df = spark.read.option("header", True).csv(filename)
    out = df.filter(df.gender=="female").filter(df.birth_year.between(1801, 1900))
    names = out.select("name").rdd.flatMap(lambda x: x).collect()[:5]
    spark.stop()
    return names

In [ ]:
prob1()

## Problem 2: Mathematicians by country

> **Problem 2.** Reads in a csv file of mathematicians, groups the
> data by country of citizenship, counts the number of occurrences of each country,
> and returns a list of the top 5 countries and their counts.

In [ ]:
# Problem 2
def prob2(filename='mathematicians.csv'):
    """Reads in a csv file of mathematicians, groups the
    data by country of citizenship, counts the number of occurrences of each country,
    and returns a list of the top 5 countries and their counts.
    """
    spark = SparkSession.builder.getOrCreate()
    df = spark.read.option("header", True).csv(filename)
    out = df.groupBy("country").count().orderBy("count", ascending=False)
    count_count = out.rdd.map(lambda x: x[:2]).collect()[:5]
    spark.stop()
    return count_count

In [ ]:
prob2()

## Problem 3: Sorted word count

> **Problem 3.** Simple sorted word count function.
>
> - Parameters: `file` (str): text file
> - Returns: `output[:5]` (list): the first five (word, count) pairs

In [ ]:
# Problem 3
def sortCount(filename='yoda.txt'):
    """Simple sorted word count function.

    Parameters:
        file (str): text file

    Returns:
        output[:5] (list): the first five (word, count) pairs
    """
    spark = SparkSession.builder.getOrCreate()
    RDD = spark.read.text(filename)\
                .rdd\
                .map(lambda x: x[0])\
                .flatMap(lambda x: x.split(" "))\
                .map(lambda x: (x,))
    df = RDD.toDF().groupBy("_1").count().orderBy("count", ascending=False)
    word_count = df.rdd.map(lambda x: x[:2]).collect()[:5]
    spark.stop()
    return word_count

In [ ]:
sortCount()

## Problem 4: Monte Carlo estimate of $\pi$

> **Problem 4.** Simple Monte-Carlo routine to estimate the area of the
> unit circle (i.e. the value of pi).
>
> - Parameters: `partitions` (int): number of partitions to run the calculation with
>   (number of partitions specifies the number of nodes to use)
> - Returns an estimated value of pi.

In [ ]:
# Problem 4
def pi(partitions=2):
    """Simple Monte-Carlo routine to estimate the area of the
    unit circle (i.e. the value of pi).

    Parameters:
        partitions (int): number of partitions to run the calculation with
            (number of partitions specifies the number of nodes to use)

    Returns an estimated value of pi.
    """
    try:
        p = 1e5
        sc = pyspark.context.SparkContext()
        l = sc.parallelize(range(int(p*partitions)), partitions)\
                    .map(lambda x: np.random.uniform(-1, 1, 2))\
                    .map(lambda x: np.linalg.norm(x) <= 1)\
                    .reduce(lambda a, b: int(a) + int(b))
        return l/(2*p)*4
    finally:
        sc.stop()

In [ ]:
pi()